# Manim Studio — Session 01

Technische Visualisierungen für *Daten, Repräsentationen und das erste Modell* (Manim Community Edition).

**Nutzung:** Jede Szenen-Zelle definiert eine `Scene` und ruft am Ende `render_scene(...)` auf. Beim Ausführen der Zelle wird die Szene gerendert, als saubere Datei nach `media/session_01/` gespeichert und direkt darunter angezeigt. Einfach die Zelle erneut ausführen, um die Datei nach Änderungen zu aktualisieren — **kein separater Export-Schritt nötig**.

**Video vs. Standbild:** Nur animieren, wenn die Bewegung die Botschaft ist (`video=True`). Alle übrigen Szenen werden als PNG gespeichert.

**Hinweis:** Titel/erklärende Texttafeln bewusst weglassen — solcher statischer Text gehört in die Präsentation. In den Szenen bleiben nur direkt beschriftende Labels (Achsen, Knoten, Gewichte …).

Szenen-Ideen inspiriert von — aber von Grund auf neu implementiert, nicht kopiert — Thomas Countz, *Calculate the Decision Boundary of a Single Perceptron*, und Joshua Thompson, *Visualizing the MLP: A Composition of Transformations*.

## Setup  *(einmal pro Kernel ausführen)*

In [2]:
import os
# In die Repo-Wurzel wechseln, egal von wo das Notebook gestartet wird
# (Notebook liegt in visualizations/; Pfade wie media/ und figures/ sind relativ zur Wurzel).
while not (os.path.isdir('figures') and os.path.isdir('sessions')) and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')

# Beim ersten Mal:  pip install manim   (+ ffmpeg)
from manim import *
import numpy as np, os, shutil
from IPython.display import Image, Video, display

config.background_color = '#0f172a'
SESSION_DIR = 'media/session_01'       # finale, saubere Dateien (eingecheckt / fuer PowerPoint)
CACHE_DIR   = 'media/.manim_cache'     # Manim-Arbeitsverzeichnis (Zwischendateien, von git ignoriert)

_QMAP = {'low': 'low_quality', 'medium': 'medium_quality',
         'high': 'high_quality', 'fourk': 'fourk_quality'}

def render_scene(scene_cls, filename, *, video=False, quality='high'):
    """Rendert `scene_cls`, speichert media/session_01/<filename>.<png|mp4> und zeigt es inline.
    Zelle erneut ausfuehren -> Datei wird aktualisiert. Kein separater Export noetig."""
    os.makedirs(SESSION_DIR, exist_ok=True)
    fmt = 'mp4' if video else 'png'
    with tempconfig({
        'media_dir': CACHE_DIR,
        'quality': _QMAP[quality],
        'format': fmt,
        'save_last_frame': not video,
        'output_file': filename,
        'verbosity': 'ERROR',
        'background_color': '#0f172a',
    }):
        scene = scene_cls()
        scene.render()
        fw = scene.renderer.file_writer
        src = fw.movie_file_path if video else fw.image_file_path
    dst = os.path.join(SESSION_DIR, f'{filename}.{fmt}')
    shutil.copy(str(src), dst)
    print(f'gespeichert -> {dst}')
    if video:
        display(Video(dst, embed=True, html_attributes='controls loop muted playsinline'))
    else:
        display(Image(filename=dst))
    return dst

import manim; print('Manim', manim.__version__)

Manim 0.20.1


### 1 · Feature Vector  *(Bild)*
Ein einzelner Datenpunkt als Pfeil im 2D-Merkmalsraum. — Folie 11

In [ ]:
class FeatureVector1(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        ax = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=6.2, y_length=6.2,
                  axis_config={'include_tip': False, 'stroke_opacity': 0.4})
        v1 = np.array([2.0, 1.5])
        vec1 = Arrow(ax.c2p(0, 0), ax.c2p(*v1), buff=0, color=BLUE_B, stroke_width=6,
                     max_tip_length_to_length_ratio=0.12)
        self.add(ax, vec1)

render_scene(FeatureVector1, 'feature_vector_1')

### 2 · Mehrere Feature Vectors  *(Bild)*
Zwei Datenpunkte als Vektoren im selben Merkmalsraum. — Folie 12

In [ ]:
class FeatureVectors2(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        ax = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=6.2, y_length=6.2,
                  axis_config={'include_tip': False, 'stroke_opacity': 0.4})
        v1 = np.array([2.0, 1.5]); v2 = np.array([-1.2, 2.0])
        vec1 = Arrow(ax.c2p(0, 0), ax.c2p(*v1), buff=0, color=BLUE_B, stroke_width=6,
                     max_tip_length_to_length_ratio=0.12)
        vec2 = Arrow(ax.c2p(0, 0), ax.c2p(*v2), buff=0, color=YELLOW, stroke_width=6,
                     max_tip_length_to_length_ratio=0.12)
        self.add(ax, vec1, vec2)

render_scene(FeatureVectors2, 'feature_vectors_2')

### 3 · Feature Space 1D  *(Bild)*
Ein Schwellwert trennt sauber, dann fallen reale Samples über die Grenze. — Folie 13

In [ ]:
class FeatureSpace1D(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        rng = np.random.default_rng(7)
        A_COL, B_COL, OK, BAD = BLUE_B, YELLOW, GREEN_B, GREY_B
        nl = NumberLine(x_range=[-5, 5, 1], length=11, include_numbers=False, stroke_opacity=0.5)
        def n1(x, j): return nl.number_to_point(x) + UP*j
        xA = rng.uniform(-3.6, -1.0, 10); xB = rng.uniform(1.0, 3.6, 10)
        jA = rng.uniform(-0.45, 0.45, 10); jB = rng.uniform(-0.45, 0.45, 10)
        dA = VGroup(*[Dot(n1(x, j), color=A_COL, radius=0.09) for x, j in zip(xA, jA)])
        dB = VGroup(*[Dot(n1(x, j), color=B_COL, radius=0.09) for x, j in zip(xB, jB)])
        thr = Line(nl.number_to_point(0)+UP*1.1, nl.number_to_point(0)+DOWN*1.1, color=OK, stroke_width=4)
        self.play(Create(nl))
        self.play(FadeIn(dA), FadeIn(dB))
        self.play(Create(thr)); self.wait(1.2)
        nxA = rng.uniform(0.3, 2.4, 4); njA = rng.uniform(-0.45, 0.45, 4)
        nxB = rng.uniform(-2.4, -0.3, 4); njB = rng.uniform(-0.45, 0.45, 4)
        noise = VGroup(*[Dot(n1(x, j), color=A_COL, radius=0.09) for x, j in zip(nxA, njA)],
                       *[Dot(n1(x, j), color=B_COL, radius=0.09) for x, j in zip(nxB, njB)])
        self.play(FadeIn(noise), thr.animate.set_color(BAD)); self.wait(1.5)

render_scene(FeatureSpace1D, 'feature_space_1d')

### 4 · Feature Space 2D  *(Bild)*
Eine Gerade trennt zwei Cluster, dann streuen Samples darüber. — Folie 13

In [ ]:
class FeatureSpace2D(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        rng = np.random.default_rng(7)
        A_COL, B_COL, OK, BAD = BLUE_B, YELLOW, GREEN_B, GREY_B
        ax = Axes(x_range=[-3.5, 3.5, 1], y_range=[-3.5, 3.5, 1], x_length=7.6, y_length=7.0,
                  axis_config={'include_tip': False, 'stroke_opacity': 0.4})
        A2 = rng.normal([-1.8, 1.8], 0.5, size=(16, 2))
        B2 = rng.normal([1.8, -1.8], 0.5, size=(16, 2))
        p2A = VGroup(*[Dot(ax.c2p(*p), color=A_COL, radius=0.08) for p in A2])
        p2B = VGroup(*[Dot(ax.c2p(*p), color=B_COL, radius=0.08) for p in B2])
        line2 = Line(ax.c2p(-3.2, -3.2), ax.c2p(3.2, 3.2), color=OK, stroke_width=4)
        self.play(Create(ax)); self.play(FadeIn(p2A), FadeIn(p2B)); self.play(Create(line2)); self.wait(1.2)
        nA2 = np.column_stack([rng.uniform(0.2, 2.6, 7), rng.uniform(-2.6, -0.2, 7)])
        nB2 = np.column_stack([rng.uniform(-2.6, -0.2, 7), rng.uniform(0.2, 2.6, 7)])
        noise2 = VGroup(*[Dot(ax.c2p(*p), color=A_COL, radius=0.08) for p in nA2],
                        *[Dot(ax.c2p(*p), color=B_COL, radius=0.08) for p in nB2])
        self.play(FadeIn(noise2), line2.animate.set_color(BAD)); self.wait(1.5)

render_scene(FeatureSpace2D, 'feature_space_2d')

### 5 · Feature Space 3D  *(Bild)*
Eine flache Ebene trennt zwei Punktwolken, dann durchdringen Samples sie. — Folie 13

In [ ]:
class FeatureSpace3D(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        rng = np.random.default_rng(7)
        A_COL, B_COL, OK, BAD = BLUE_B, YELLOW, GREEN_B, GREY_B
        ex = np.array([1.0, 0.0]); ey = np.array([0.0, 1.0]); ez = np.array([0.45, 0.32]); SC = 2.0
        def P(p):
            s = SC*(p[0]*ex + p[1]*ey + p[2]*ez)
            return np.array([s[0], s[1], 0.0])
        corners = np.array([[sx, sy, sz] for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)], float)
        edges = VGroup(*[Line(P(corners[i]), P(corners[j]), color='#3a4a63', stroke_width=2)
                         for i in range(8) for j in range(i+1, 8)
                         if np.sum(np.abs(corners[i]-corners[j]) > 1e-9) == 1])
        v1 = np.array([1, -1, 0])/np.sqrt(2); v2 = np.array([1, 1, -2])/np.sqrt(6)
        patch = Polygon(*[P(a*v1 + b*v2) for a, b in [(-1.15, -1.15), (1.15, -1.15), (1.15, 1.15), (-1.15, 1.15)]],
                        stroke_color=OK, stroke_width=3).set_fill(OK, opacity=0.18)
        A3 = rng.normal([-0.55, -0.55, -0.55], 0.18, size=(16, 3))
        B3 = rng.normal([0.55, 0.55, 0.55], 0.18, size=(16, 3))
        p3A = VGroup(*[Dot(P(p), color=A_COL, radius=0.09) for p in A3])
        p3B = VGroup(*[Dot(P(p), color=B_COL, radius=0.09) for p in B3])
        self.play(Create(edges)); self.play(FadeIn(p3A), FadeIn(p3B)); self.play(FadeIn(patch)); self.wait(1.2)
        nA3 = rng.uniform([0.1, 0.1, 0.1], [1.0, 1.0, 1.0], size=(7, 3))
        nB3 = rng.uniform([-1.0, -1.0, -1.0], [-0.1, -0.1, -0.1], size=(7, 3))
        noise3 = VGroup(*[Dot(P(p), color=A_COL, radius=0.09) for p in nA3],
                        *[Dot(P(p), color=B_COL, radius=0.09) for p in nB3])
        self.play(FadeIn(noise3), patch.animate.set_stroke(BAD).set_fill(BAD, opacity=0.12)); self.wait(1.5)

render_scene(FeatureSpace3D, 'feature_space_3d')

### 6 · Standardisierung  *(Bild)*
Gestreckte Loss-Konturen (langsamer Abstieg) werden nach Standardisierung auf Einheitsvarianz rund. — Folie 14

In [ ]:
class Standardization(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        rng = np.random.default_rng(0)
        pts = rng.normal(0, 1, size=(60, 2)) * np.array([3.0, 0.35])
        axL = Axes(x_range=[-8, 8, 2], y_range=[-3, 3, 1], x_length=6, y_length=4,
                   axis_config={'include_tip': False, 'stroke_opacity': 0.4}).to_edge(LEFT, buff=0.5).shift(DOWN*0.2)
        cloud = VGroup(*[Dot(axL.c2p(*p), color=BLUE_B, radius=0.05) for p in pts])
        ell = VGroup(*[Ellipse(width=w*2.4, height=w*0.45, color=YELLOW, stroke_opacity=0.5).move_to(axL.c2p(0, 0)) for w in (1, 2, 3)])
        axR = Axes(x_range=[-3, 3, 1], y_range=[-3, 3, 1], x_length=4.4, y_length=4.4,
                   axis_config={'include_tip': False, 'stroke_opacity': 0.4}).to_edge(RIGHT, buff=0.5).shift(DOWN*0.2)
        z = (pts - pts.mean(0)) / pts.std(0)
        cloudZ = VGroup(*[Dot(axR.c2p(*p), color=BLUE_B, radius=0.05) for p in z])
        circ = VGroup(*[Circle(radius=r, color=GREEN_B, stroke_opacity=0.6).move_to(axR.c2p(0, 0)) for r in (0.8, 1.6, 2.4)])
        self.add(axL, cloud, ell, axR, cloudZ, circ)

render_scene(Standardization, 'standardization')

### 7 · Das Perceptron (Forward-Pass)  *(Video)*
Gewichtete Summe Σ → Aktivierung φ → Ausgabe, für mehrere Eingaben. — Folien 17–22

In [ ]:
class Perceptron(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        EDGE, LBL, DIM = '#3a4a63', '#9cc2f5', '#475569'

        # --- feste Parameter --------------------------------------------------
        w = np.array([1.0, 0.8, -1.2]); b = -0.3
        X = [np.array(v) for v in ([1, 0, 1], [1, 1, 0], [0, 1, 1], [1, 1, 1])]

        # --- Geometrie --------------------------------------------------------
        in_x, sum_x, act_x, out_x = -5.2, -0.4, 2.4, 5.2
        in_ys = [2.0, 0.7, -0.6]; bias_y = -2.2; r = 0.42

        in_nodes = VGroup(*[Circle(radius=r, color=BLUE_B, stroke_width=3).set_fill(BLUE_B, 0.12).move_to([in_x, y, 0]) for y in in_ys])
        in_labels = VGroup(*[Text(t, font_size=22).next_to(n, LEFT, buff=0.25)
                             for t, n in zip(['x₁', 'x₂', 'x₃'], in_nodes)])
        bias_node = Circle(radius=r, color=YELLOW, stroke_width=3).set_fill(YELLOW, 0.12).move_to([in_x, bias_y, 0])
        bias_val = Text('1', font_size=22).move_to(bias_node)
        bias_label = Text('Bias', font_size=20, color=YELLOW).next_to(bias_node, LEFT, buff=0.25)

        sum_node = Circle(radius=0.6, color=WHITE, stroke_width=3).move_to([sum_x, 0, 0])
        sum_sym = Text('Σ', font_size=40).move_to(sum_node)

        act = RoundedRectangle(corner_radius=0.12, width=1.5, height=1.5, color=WHITE, stroke_width=2).move_to([act_x, 0, 0])
        c = np.array([act_x, 0, 0])
        step = VGroup(
            Line(c + [-0.55, 0, 0], c + [0.55, 0, 0], color=EDGE, stroke_width=1.5),
            Line(c + [-0.5, -0.32, 0], c + [0, -0.32, 0], color=BLUE_B, stroke_width=4),
            Line(c + [0, -0.32, 0], c + [0, 0.32, 0], color=BLUE_B, stroke_width=4),
            Line(c + [0, 0.32, 0], c + [0.5, 0.32, 0], color=BLUE_B, stroke_width=4),
        )
        act_label = Text('Sprung', font_size=20).next_to(act, DOWN, buff=0.18)

        out_node = Circle(radius=r, color=WHITE, stroke_width=3).move_to([out_x, 0, 0])
        out_label = Text('y', font_size=24).next_to(out_node, RIGHT, buff=0.25)

        srcs = list(in_nodes) + [bias_node]; weights = list(w) + [b]
        edges = VGroup(); wlabels = VGroup()
        for node, wt in zip(srcs, weights):
            edge = Line(node.get_right(), sum_node.get_left(), color=EDGE, stroke_width=2, stroke_opacity=0.7)
            edges.add(edge)
            wlabels.add(Text(f'{wt:+.1f}', font_size=18, color=LBL).move_to(edge.point_from_proportion(0.28) + np.array([0, 0.22, 0])))
        e_sa = Line(sum_node.get_right(), act.get_left(), color=EDGE, stroke_width=2, stroke_opacity=0.7)
        e_ao = Line(act.get_right(), out_node.get_left(), color=EDGE, stroke_width=2, stroke_opacity=0.7)

        self.play(LaggedStart(*[GrowFromCenter(n) for n in [*in_nodes, bias_node]], lag_ratio=0.1),
                  FadeIn(in_labels), FadeIn(bias_label), FadeIn(bias_val))
        self.play(Create(edges), FadeIn(wlabels))
        self.play(GrowFromCenter(sum_node), Write(sum_sym))
        self.play(Create(e_sa), FadeIn(act), Create(step), FadeIn(act_label))
        self.play(Create(e_ao), GrowFromCenter(out_node), FadeIn(out_label))

        def make_in(x): return VGroup(*[Text(str(int(xi)), font_size=24).move_to(n) for xi, n in zip(x, in_nodes)])
        state = {
            'in':  make_in(X[0]).set_opacity(0),
            'z':   Text('z = +0.00', font_size=22, color=LBL).next_to(sum_node, DOWN, buff=0.9).set_opacity(0),
            'out': Text('0', font_size=26).move_to(out_node).set_opacity(0),
        }
        self.add(state['in'], state['z'], state['out'])

        def flash(line):
            return ShowPassingFlash(line.copy().set_color(YELLOW).set_stroke(width=5), time_width=0.6)

        def show_input(x):
            z = float(w @ x + b); y = 1 if z >= 0 else 0
            new_in = make_in(x)
            self.play(ReplacementTransform(state['in'], new_in), run_time=0.4); state['in'] = new_in
            self.play(*[flash(e) for e in edges], run_time=0.9)
            new_z = Text(f'z = {z:+.2f}', font_size=22, color=LBL).next_to(sum_node, DOWN, buff=0.9)
            self.play(ReplacementTransform(state['z'], new_z), run_time=0.3); state['z'] = new_z
            self.play(flash(e_sa), run_time=0.6)
            self.play(flash(e_ao), run_time=0.6)
            col = GREEN_B if y == 1 else DIM
            new_out = Text(str(y), font_size=26).move_to(out_node)
            self.play(out_node.animate.set_fill(col, opacity=0.9).set_stroke(col),
                      ReplacementTransform(state['out'], new_out), run_time=0.4); state['out'] = new_out
            self.wait(0.8)

        for x in X:
            show_input(x)
        self.wait(1.0)

render_scene(Perceptron, 'perceptron_forward', video=True, quality='medium')   # quality='high' fuer 1080p

### 8 · Multi-Layer-Perceptron (Forward-Pass)  *(Video)*
Signal fließt durch ein 6→2→1-Netz: Eingaben → Hidden Layer (Σ+b, ReLU) → Ausgabe (Σ+b, Sprung). Aktive Einheiten leuchten grün auf. Zwei Eingabemuster zeigen die Ausgaben y = 0 und y = 1. — MLP-Folien

In [ ]:
EDGE, LBL, DIM = '#3a4a63', '#9cc2f5', '#475569'

def _relu_glyph(box):
    """kleine ReLU-Kurve (flach, dann ansteigend) rechts im Neuron-Kasten."""
    c = box.get_center() + RIGHT*0.34
    return VGroup(Line(c + [-0.30, -0.18, 0], c + [0.02, -0.18, 0], color=BLUE_B, stroke_width=4),
                  Line(c + [0.02, -0.18, 0], c + [0.30, 0.20, 0], color=BLUE_B, stroke_width=4))

def _step_glyph(box):
    """kleine Sprung-Kurve (Heaviside) rechts im Neuron-Kasten."""
    c = box.get_center() + RIGHT*0.34
    return VGroup(Line(c + [-0.30, -0.18, 0], c + [0.0, -0.18, 0], color=BLUE_B, stroke_width=4),
                  Line(c + [0.0, -0.18, 0], c + [0.0, 0.20, 0], color=BLUE_B, stroke_width=4),
                  Line(c + [0.0, 0.20, 0], c + [0.30, 0.20, 0], color=BLUE_B, stroke_width=4))

def _neuron_box(pos, glyph_fn):
    box = RoundedRectangle(corner_radius=0.12, width=1.55, height=1.0,
                           color='#334155', stroke_width=2).move_to(pos)
    sym = Text('Σ+b', font_size=24).move_to(box).shift(LEFT*0.32)
    div = Line(box.get_top()+DOWN*0.12, box.get_bottom()+UP*0.12,
               color='#334155', stroke_width=1.5).move_to(box.get_center()+RIGHT*0.04)
    grp = VGroup(box, sym, div, glyph_fn(box)); grp.box = box
    return grp


class MLP(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'

        # --- Architektur & feste Parameter -----------------------------------
        rng = np.random.default_rng(3)
        W1 = rng.normal(0, 1.1, size=(2, 6)); b1 = np.array([-0.4, -0.2])
        W2 = np.array([1.3, 1.1]); b2 = -0.9
        in_x, hid_x, out_x = -5.4, 0.0, 4.4
        in_ys = np.linspace(2.6, -2.6, 6); hid_ys = [1.35, -1.35]

        in_nodes = VGroup(*[Circle(radius=0.34, color=BLUE_B, stroke_width=3).move_to([in_x, y, 0]) for y in in_ys])
        in_labels = VGroup(*[Text(f'x{chr(0x2080+i+1)}', font_size=20, color=LBL).next_to(n, LEFT, buff=0.2)
                             for i, n in enumerate(in_nodes)])
        hid = VGroup(*[_neuron_box([hid_x, y, 0], _relu_glyph) for y in hid_ys])
        out = _neuron_box([out_x, 0, 0], _step_glyph)
        out_label = Text('y', font_size=26).next_to(out, RIGHT, buff=0.3)
        caps = VGroup(Text('Eingabe', font_size=22, color=LBL).move_to([in_x, -3.35, 0]),
                      Text('Hidden Layer', font_size=22, color=LBL).move_to([hid_x, -3.35, 0]),
                      Text('Ausgabe', font_size=22, color=LBL).move_to([out_x, -3.35, 0]))

        e_ih = VGroup(*[Line(n.get_right(), hb.box.get_left(), color=EDGE, stroke_width=1.6, stroke_opacity=0.6)
                        for n in in_nodes for hb in hid])
        e_ho = VGroup(*[Line(hb.box.get_right(), out.box.get_left(), color=EDGE, stroke_width=2.2, stroke_opacity=0.7)
                        for hb in hid])

        # --- Aufbau ----------------------------------------------------------
        self.play(LaggedStart(*[GrowFromCenter(n) for n in in_nodes], lag_ratio=0.08), FadeIn(in_labels), run_time=1.2)
        self.play(Create(e_ih), run_time=1.0)
        self.play(LaggedStart(*[FadeIn(hb, scale=0.85) for hb in hid], lag_ratio=0.2), run_time=0.9)
        self.play(Create(e_ho), FadeIn(out), FadeIn(out_label), run_time=0.9)
        self.play(FadeIn(caps), run_time=0.6); self.wait(0.4)

        # --- Forward-Pass ----------------------------------------------------
        in_vals = {}
        def set_inputs(x):
            anims = []
            for i, (n, xi) in enumerate(zip(in_nodes, x)):
                t = Text(str(int(xi)), font_size=22).move_to(n)
                anims.append(ReplacementTransform(in_vals[i], t) if i in in_vals else FadeIn(t))
                in_vals[i] = t
                anims.append(n.animate.set_fill(BLUE_B, 0.85 if xi == 1 else 0.0))
            self.play(*anims, run_time=0.5)

        def pulse(edges):
            self.play(*[ShowPassingFlash(e.copy().set_color(YELLOW).set_stroke(width=4), time_width=0.55) for e in edges], run_time=0.9)

        def light(grp, active):
            col = GREEN_B if active else DIM
            self.play(grp.box.animate.set_stroke(col).set_fill(col, 0.18 if active else 0.0), run_time=0.4)

        def forward(x):
            set_inputs(x); pulse(e_ih)
            h = np.maximum(W1 @ x + b1, 0)
            for hb, hv in zip(hid, h):
                light(hb, hv > 0)
            self.wait(0.2); pulse(e_ho)
            y = 1 if (W2 @ h + b2) >= 0 else 0
            light(out, y == 1)
            self.play(out_label.animate.set_color(GREEN_B if y == 1 else DIM), run_time=0.3)
            self.wait(0.9)

        forward(np.array([0, 0, 0, 0, 1, 0]))   # eine verborgene Einheit aktiv -> y = 0
        for hb in hid:
            self.play(hb.box.animate.set_stroke('#334155').set_fill(BLACK, 0.0), run_time=0.25)
        self.play(out.box.animate.set_stroke('#334155').set_fill(BLACK, 0.0),
                  out_label.animate.set_color(WHITE), run_time=0.25)
        forward(np.array([1, 0, 0, 1, 1, 1]))   # beide verborgenen Einheiten aktiv -> y = 1
        self.wait(1.2)

render_scene(MLP, 'mlp', video=True, quality='medium')   # quality='high' fuer 1080p

### 9 · Vom Bild zum MLP  *(Video)*
Das ganze Foto wird in Pixel zerlegt — jedes Pixel ist ein Eingabeneuron. Die Pixel der Katze werden hervorgehoben (Rest abgedunkelt) und lösen sich fließend aus dem Bild in die Eingabeschicht. Danach arbeitet das MLP wie zuvor: Eingabe → Hidden Layer (Σ+b, ReLU) → Ausgabe (Klassen *Hund* / *Katze*, hier *Katze*). Benötigt `figures/cat_dog_scene.png`. — Bild-Klassifikations-Folien

In [3]:
from PIL import Image as PILImage
EDGE, LBL, DIM = '#3a4a63', '#9cc2f5', '#475569'
_GW, _GH = 24, 18
_CAT_COLS = range(13, 20) 
_CAT_ROWS = range(5, 13)   

def _hexc(rgb):
    return '#%02x%02x%02x' % (int(rgb[0]), int(rgb[1]), int(rgb[2]))

def _relu_glyph(box):
    c = box.get_center() + RIGHT*0.34
    return VGroup(Line(c+[-0.30,-0.18,0], c+[0.02,-0.18,0], color=BLUE_B, stroke_width=4),
                  Line(c+[0.02,-0.18,0], c+[0.30,0.20,0], color=BLUE_B, stroke_width=4))

def _neuron_box(pos):
    box = RoundedRectangle(corner_radius=0.12, width=1.55, height=1.0, color='#334155', stroke_width=2).move_to(pos)
    sym = Text('Σ+b', font_size=24).move_to(box).shift(LEFT*0.32)
    div = Line(box.get_top()+DOWN*0.12, box.get_bottom()+UP*0.12, color='#334155', stroke_width=1.5).move_to(box.get_center()+RIGHT*0.04)
    grp = VGroup(box, sym, div, _relu_glyph(box)); grp.box = box
    return grp


class ImageMLP(Scene):
    def construct(self):
        self.camera.background_color = '#0f172a'
        full = PILImage.open('figures/cat_dog_scene.png').convert('RGB')
        small = np.array(full.resize((_GW, _GH)))

        # ---- Foto -> Pixel-Mosaik (ganzes Bild) -----------------------------
        photo = ImageMobject('figures/cat_dog_scene.png'); photo.height = 4.7; photo.move_to([-1.7, 0.3, 0])
        s = photo.width / _GW
        cx, cy = photo.get_center()[0], photo.get_center()[1]
        W, H = _GW*s, _GH*s

        def cell_center(i, j):
            return np.array([cx - W/2 + (i+0.5)*s, cy + H/2 - (j+0.5)*s, 0.0])

        # Pixel ohne Rand (stroke_width=0) -> keine Kantenartefakte
        squares = VGroup()
        for j in range(_GH):
            for i in range(_GW):
                squares.add(Square(side_length=s, stroke_width=0,
                                   fill_color=_hexc(small[j, i]), fill_opacity=1.0).move_to(cell_center(i, j)))

        def sq_at(i, j):
            return squares[j*_GW + i]

        cat_set = {(i, j) for i in _CAT_COLS for j in _CAT_ROWS}

        # Pixel deckend ueber das Foto einblenden (kein Cross-Fade -> keine Naht-Linien)
        self.play(FadeIn(photo, run_time=1.0)); self.wait(0.3)
        self.play(FadeIn(squares, lag_ratio=0.003, run_time=1.6)); self.remove(photo); self.wait(0.3)

        # ---- Katzen-Pixel hervorheben (Rest abdunkeln) ----------------------
        cl, rl = list(_CAT_COLS), list(_CAT_ROWS)
        region = Rectangle(width=len(cl)*s, height=len(rl)*s, stroke_color=YELLOW, stroke_width=3, fill_opacity=0)
        region.move_to((cell_center(cl[0], rl[0]) + cell_center(cl[-1], rl[-1]))/2)
        non_cat = [sq_at(i, j) for j in range(_GH) for i in range(_GW) if (i, j) not in cat_set]
        self.play(*[sq.animate.set_opacity(0.18) for sq in non_cat], Create(region), run_time=1.1)
        self.wait(0.6)

        # ---- fluider Übergang: Katzen-Pixel -> Eingabeneuronen --------------
        self.play(VGroup(squares, region).animate.scale(0.52).move_to([-5.0, 0.3, 0]), run_time=1.0)

        col_x = -2.7; col_ys = np.linspace(2.7, -2.1, 8)
        sample_ij = [(15, 7), (18, 7), (14, 9), (19, 9), (16, 10), (13, 12), (17, 13), (15, 14)]
        in_neurons = VGroup(*[Circle(radius=0.15, stroke_color='#9cc2f5', stroke_width=2,
                                     fill_color=_hexc(small[j, i]), fill_opacity=1.0).move_to([col_x, y, 0])
                              for (i, j), y in zip(sample_ij, col_ys)])
        dots = Text('⋮', font_size=30, color=LBL).move_to([col_x, -2.7, 0])

        flyers = [sq_at(i, j).copy().set_opacity(1.0) for (i, j) in sample_ij]
        self.play(LaggedStart(*[ReplacementTransform(f, n) for f, n in zip(flyers, in_neurons)],
                              lag_ratio=0.14), run_time=2.0)
        self.play(FadeIn(dots), run_time=0.3)

        hid = VGroup(*[_neuron_box([0.9, y, 0]) for y in (1.7, 0.0, -1.7)])
        out_neurons = VGroup(*[Circle(radius=0.32, color='#334155', stroke_width=3).move_to([4.5, y, 0]) for y in (1.0, -1.0)])
        out_labels = VGroup(Text('Hund', font_size=24, color=WHITE).next_to(out_neurons[0], RIGHT, buff=0.25),
                            Text('Katze', font_size=24, color=WHITE).next_to(out_neurons[1], RIGHT, buff=0.25))

        e_ih = VGroup(*[Line(n.get_right(), hb.box.get_left(), color=EDGE, stroke_width=1.3, stroke_opacity=0.5)
                        for n in in_neurons for hb in hid])
        e_ho = VGroup(*[Line(hb.box.get_right(), o.get_left(), color=EDGE, stroke_width=1.8, stroke_opacity=0.6)
                        for hb in hid for o in out_neurons])

        caps = VGroup(Text('Eingabe', font_size=22, color=LBL).move_to([col_x, -3.4, 0]),
                      Text('Hidden Layer', font_size=22, color=LBL).move_to([0.9, -3.4, 0]),
                      Text('Ausgabe', font_size=22, color=LBL).move_to([4.7, -3.4, 0]))

        self.play(Create(e_ih), run_time=1.0)
        self.play(LaggedStart(*[FadeIn(hb, scale=0.85) for hb in hid], lag_ratio=0.2), run_time=0.9)
        self.play(Create(e_ho), LaggedStart(*[GrowFromCenter(o) for o in out_neurons], lag_ratio=0.2),
                  FadeIn(out_labels), run_time=0.9)
        self.play(FadeIn(caps), run_time=0.5); self.wait(0.3)

        # ---- Forward-Pass ----------------------------------------------------
        def pulse(edges, rt=0.9):
            self.play(*[ShowPassingFlash(e.copy().set_color(YELLOW).set_stroke(width=3.5), time_width=0.55) for e in edges], run_time=rt)

        pulse(e_ih)
        for hb, a in zip(hid, [True, True, False]):
            col = GREEN_B if a else DIM
            self.play(hb.box.animate.set_stroke(col).set_fill(col, 0.18 if a else 0.0), run_time=0.35)
        self.wait(0.2); pulse(e_ho)
        self.play(out_neurons[1].animate.set_stroke(GREEN_B).set_fill(GREEN_B, 0.85),
                  out_labels[1].animate.set_color(GREEN_B),
                  out_neurons[0].animate.set_stroke(DIM), out_labels[0].animate.set_color(DIM), run_time=0.6)
        self.wait(1.4)

render_scene(ImageMLP, 'image_mlp', video=True, quality='medium')   # quality='high' fuer 1080p

gespeichert -> media/session_01\image_mlp.mp4


'media/session_01\\image_mlp.mp4'